In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyscf
import pyscf.cc
import pyscf.fci
import pyscf.data.elements
import ffsim
from scipy.sparse.linalg import LinearOperator

from xquces.hamiltonians import MolecularHamiltonianLinearOperator
from xquces.states import hartree_fock_state
from xquces.ucj.init import UCJBalancedDFSeed
from xquces.ucj.parameterization import GaugeFixedUCJSpinBalancedParameterization


def linear_operator_from_xquces_hamiltonian(ham):
    dim = ham.matvec(hartree_fock_state(ham.norb, ham.nelec)).size

    def matvec(v):
        v = np.asarray(v, dtype=np.complex128).reshape(-1)
        return ham.matvec(v) + ham.ecore * v

    def matmat(vs):
        vs = np.asarray(vs, dtype=np.complex128)
        return np.column_stack([matvec(vs[:, j]) for j in range(vs.shape[1])])

    return LinearOperator(
        shape=(dim, dim),
        matvec=matvec,
        matmat=matmat,
        dtype=np.complex128,
    )


def default_active_space(mf, n_frozen=None):
    if n_frozen is None:
        n_frozen = pyscf.data.elements.chemcore(mf.mol)
    return list(range(n_frozen, mf.mo_coeff.shape[1]))


def frozen_from_active_space(mf, active_space):
    return sorted(set(range(mf.mo_coeff.shape[1])) - set(active_space))


atom = "N 0 0 0; N 0 0 1.2"
basis = "sto-6g"
n_reps = 2
tol = 1e-8
optimize_df = False
maxiter = 10

mol = pyscf.gto.M(
    atom=atom,
    basis=basis,
    spin=0,
    unit="Angstrom",
)
mf = pyscf.scf.RHF(mol).run()

active_space = default_active_space(mf)
frozen = frozen_from_active_space(mf, active_space)

cc = pyscf.cc.CCSD(mf, frozen=frozen).run()

ham_xq = MolecularHamiltonianLinearOperator.from_scf(
    mf,
    active_space=active_space,
)
ham_lm = linear_operator_from_xquces_hamiltonian(ham_xq)

norb = ham_xq.norb
nelec = ham_xq.nelec
nocc = nelec[0]

reference_vec_xq = hartree_fock_state(norb, nelec)
reference_vec_ff = ffsim.hartree_fock_state(norb, nelec)

e_fci, _ = pyscf.fci.direct_spin1.kernel(
    ham_xq.h1,
    ham_xq.eri,
    norb,
    nelec,
    ecore=ham_xq.ecore,
)

ansatz_seed_xq = UCJBalancedDFSeed(
    t2=cc.t2,
    t1=cc.t1,
    n_reps=n_reps,
    tol=tol,
    optimize=optimize_df,
).build_ansatz()

parametrization_xq = GaugeFixedUCJSpinBalancedParameterization(
    norb=norb,
    nocc=nocc,
    n_layers=ansatz_seed_xq.n_layers,
    with_final_orbital_rotation=ansatz_seed_xq.final_orbital_rotation is not None,
)

x0_xq = parametrization_xq.parameters_from_ansatz(ansatz_seed_xq)
params_to_vec_xq = parametrization_xq.params_to_vec(reference_vec_xq, nelec)

moldata = ffsim.MolecularData.from_scf(mf, active_space=active_space)

ucj_ff = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    cc.t2,
    t1=cc.t1,
    n_reps=n_reps,
    tol=tol,
    optimize=optimize_df,
)

x0_ff = ucj_ff.to_parameters()
with_final_orbital_rotation_ff = ucj_ff.final_orbital_rotation is not None

ham_ff = ffsim.linear_operator(
    moldata.hamiltonian,
    norb=moldata.norb,
    nelec=moldata.nelec,
)

def params_to_vec_ff(params):
    op = ffsim.UCJOpSpinBalanced.from_parameters(
        np.asarray(params, dtype=np.float64),
        norb=norb,
        n_reps=ucj_ff.n_reps,
        interaction_pairs=None,
        with_final_orbital_rotation=with_final_orbital_rotation_ff,
    )
    return ffsim.apply_unitary(
        reference_vec_ff,
        op,
        norb=norb,
        nelec=nelec,
        copy=True,
    )

e_hf = ham_xq.expectation(reference_vec_xq)

vec_seed_direct_xq = ansatz_seed_xq.apply(reference_vec_xq, nelec=nelec, copy=True)
e_seed_direct_xq = ham_xq.expectation(vec_seed_direct_xq)

vec0_xq = params_to_vec_xq(x0_xq)
e0_xq = ham_xq.expectation(vec0_xq)

vec0_ff = params_to_vec_ff(x0_ff)
e0_ff = float(np.vdot(vec0_ff, ham_ff @ vec0_ff).real)

n_params_xq = parametrization_xq.n_params
n_params_ff = len(x0_ff)

print("active_space =", active_space)
print("frozen =", frozen)
print("norb =", norb)
print("nelec =", nelec)
print("FCI energy =", e_fci)
print("HF energy =", e_hf)
print()

print("xquces direct seed energy =", e_seed_direct_xq)
print("xquces roundtrip seed energy =", e0_xq)
print("xquces roundtrip seed error =", abs(e0_xq - e_fci))
print()

print("xquces gauge-fixed n_layers =", ansatz_seed_xq.n_layers)
print("xquces gauge-fixed n_params =", n_params_xq)
print()

print("ffsim n_reps =", ucj_ff.n_reps)
print("ffsim has final orbital rotation =", with_final_orbital_rotation_ff)
print("ffsim n_params =", n_params_ff)
print("ffsim seed energy =", e0_ff)
print("ffsim seed error =", abs(e0_ff - e_fci))
print()

print("param ratio ffsim/xquces =", n_params_ff / n_params_xq)

trace_xq = []
trace_ff = []

def callback_xq(intermediate_result):
    e = float(intermediate_result.fun)
    trace_xq.append(e)
    print(
        f"[xquces] iter {len(trace_xq):3d}  "
        f"E = {e:.12f}  "
        f"|E - E_FCI| = {abs(e - e_fci):.12e}"
    )

def callback_ff(intermediate_result):
    e = float(intermediate_result.fun)
    trace_ff.append(e)
    print(
        f"[ffsim ] iter {len(trace_ff):3d}  "
        f"E = {e:.12f}  "
        f"|E - E_FCI| = {abs(e - e_fci):.12e}"
    )

res_xq = ffsim.optimize.minimize_linear_method(
    params_to_vec_xq,
    ham_lm,
    x0_xq,
    maxiter=maxiter,
    ftol=1e-16,
    callback=callback_xq,
)

res_ff = ffsim.optimize.minimize_linear_method(
    params_to_vec_ff,
    ham_ff,
    x0_ff,
    maxiter=maxiter,
    ftol=1e-16,
    callback=callback_ff,
)

x_opt_xq = np.asarray(res_xq.x, dtype=np.float64)
vec_opt_xq = params_to_vec_xq(x_opt_xq)
e_opt_xq = ham_xq.expectation(vec_opt_xq)

x_opt_ff = np.asarray(res_ff.x, dtype=np.float64)
vec_opt_ff = params_to_vec_ff(x_opt_ff)
e_opt_ff = float(np.vdot(vec_opt_ff, ham_ff @ vec_opt_ff).real)

print()
print("Optimization finished")
print("xquces success =", res_xq.success)
print("xquces message =", res_xq.message)
print("xquces iterations =", res_xq.nit)
print("xquces final energy =", e_opt_xq)
print("xquces final error =", abs(e_opt_xq - e_fci))
print()

print("ffsim success =", res_ff.success)
print("ffsim message =", res_ff.message)
print("ffsim iterations =", res_ff.nit)
print("ffsim final energy =", e_opt_ff)
print("ffsim final error =", abs(e_opt_ff - e_fci))

iters_xq = np.arange(1, len(trace_xq) + 1)
iters_ff = np.arange(1, len(trace_ff) + 1)
err_xq = np.abs(np.asarray(trace_xq) - e_fci)
err_ff = np.abs(np.asarray(trace_ff) - e_fci)

plt.figure(figsize=(7, 5))
plt.semilogy(iters_xq, err_xq, marker="o", label=f"xquces gauge-fixed ({n_params_xq} params)")
plt.semilogy(iters_ff, err_ff, marker="s", label=f"ffsim UCJOpSpinBalanced ({n_params_ff} params)")
plt.xlabel("LM iteration")
plt.ylabel(r"$|E - E_{\mathrm{FCI}}|$")
plt.title("Error from active-space FCI vs LM iteration")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

/home/david/repos/xquces/.venv/lib/python3.12/site-packages/numpy/_core/getlimits.py:552: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


converged SCF energy = -108.535614528761
E(CCSD) = -108.7208078632474  E_corr = -0.1851933344863977
active_space = [2, 3, 4, 5, 6, 7, 8, 9]
frozen = [0, 1]
norb = 8
nelec = (5, 5)
FCI energy = -108.7268549019409
HF energy = -108.53561452876107

xquces direct seed energy = -108.42132823564877
xquces roundtrip seed energy = -108.4213282356491
xquces roundtrip seed error = 0.30552666629179726

xquces gauge-fixed n_layers = 2
xquces gauge-fixed n_params = 320

ffsim n_reps = 2
ffsim has final orbital rotation = True
ffsim n_params = 336
ffsim seed energy = -108.6789761832217
ffsim seed error = 0.047878718719189806

param ratio ffsim/xquces = 1.05
[xquces] iter   1  E = -108.611736449098  |E - E_FCI| = 1.151184528431e-01
[xquces] iter   2  E = -108.628854398129  |E - E_FCI| = 9.800050381203e-02
